Ce script est conçu pour traiter un fichier CSV contenant des coordonnées GPS (latitude et longitude) et calculer la moyenne de l'intensité lumineuse nocturne dans une zone carrée de 16 km² autour de chaque point GPS. Les données des intensités lumineuses sont issues d'un fichier raster .tif. Les résultats sont ajoutés dans une nouvelle colonne du fichier CSV, qui est ensuite sauvegardé dans un répertoire spécifique.

Le script commence par définir les chemins des fichiers d'entrée (le fichier CSV contenant les points GPS et le fichier raster contenant les données d'intensité lumineuse) et un chemin de sortie où les résultats seront sauvegardés. Il vérifie si le répertoire de sortie existe, et le crée si nécessaire, pour garantir que les fichiers traités soient correctement organisés. Ensuite, il charge le fichier CSV dans un DataFrame Pandas pour permettre un traitement facile des données.

À l'aide de la bibliothèque Rasterio, le script ouvre le fichier raster et extrait ses métadonnées, y compris la transformation affine qui permet de convertir les coordonnées géographiques (longitude et latitude) en indices de pixels dans l'image raster. Une fonction calculate_mean_radiance est définie pour calculer la moyenne de l'intensité lumineuse dans une fenêtre de 8x8 pixels (équivalent à 4 km x 4 km avec une résolution de 0,5 km par pixel) autour d'un point donné. Cette fonction vérifie si les pixels sélectionnés se trouvent à l'intérieur des limites de l'image raster et gère les valeurs manquantes (no_data) en les excluant du calcul.

La fonction est ensuite appliquée à chaque ligne du DataFrame, et les résultats sont stockés dans une nouvelle colonne appelée radiance_16_km^2. Enfin, le DataFrame mis à jour est sauvegardé sous forme de fichier CSV dans le répertoire spécifié. Le script imprime un message confirmant que le fichier a été traité avec succès, incluant le chemin du fichier sauvegardé. Ce workflow est utile pour l'analyse spatiale, notamment dans le contexte d'études sur la relation entre les intensités lumineuses nocturnes et des variables socio-économiques.

In [ ]:
!pip install rasterio

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.windows import Window

# Charger le fichier CSV
csv_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled.csv"
df = pd.read_csv(csv_file_path)

# Charger le fichier TIFF (Nighttime light)
tif_file_path = r"D:\wealth_predict_sentinel\Data\downloaded\VNL_npp_2023_global_vcmslcfg_v2_c202402081600.average.dat.tif"
with rasterio.open(tif_file_path) as dataset:
    raster_data = dataset.read(1)
    transform = dataset.transform
    nodata_value = dataset.nodata

    # Transformation inverse des coordonnées GPS en indices de pixels
    def gps_to_pixel(lon, lat, transform):
        col, row = ~transform * (lon, lat)
        return int(col), int(row)

    # Calculer la moyenne de radiance dans une fenêtre 8x8 pixels
    def calculate_mean_radiance(lon, lat):
        try:
            # Transformer les coordonnées GPS en indices de pixels
            col, row = gps_to_pixel(lon, lat, transform)

            # Définir une fenêtre 8x8 pixels autour du point central
            window = Window(col - 4, row - 4, 8, 8)

            # Vérifier si la fenêtre dépasse les limites du raster
            if (window.col_off < 0 or window.row_off < 0 or
                window.col_off + window.width > raster_data.shape[1] or
                window.row_off + window.height > raster_data.shape[0]):
                return np.nan

            # Extraire les données dans la fenêtre
            data = raster_data[window.row_off:window.row_off + window.height,
                               window.col_off:window.col_off + window.width]

            # Remplacer les valeurs no_data par NaN
            if nodata_value is not None:
                data = data.astype(float)
                data[data == nodata_value] = np.nan

            # Calculer et retourner la moyenne
            return np.nanmean(data)
        except Exception as e:
            print(f"Erreur pour les coordonnées ({lat}, {lon}): {e}")
            return np.nan

    # Appliquer la fonction sur chaque point GPS
    df['radiance_16_km^2'] = df.apply(lambda row: calculate_mean_radiance(row['longitude'], row['latitude']), axis=1)

# Sauvegarder les résultats
output_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance.csv"
df.to_csv(output_file_path, index=False)

print(f"Fichier sauvegardé avec succès dans : {output_file_path}")


In [ ]:
import pandas as pd
pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance.csv")

- Ce script analyse la relation entre deux variables numériques, radiance_16_km^2 et radiance, présentes dans un fichier CSV. Il calcule leur corrélation et produit une visualisation graphique de cette relation sous la forme d'un graphe de dispersion avec une ligne de régression linéaire pour indiquer la tendance.
- Le script commence par charger les données depuis un fichier CSV à l'aide de la bibliothèque pandas. Il extrait les colonnes radiance_16_km^2 et radiance pour effectuer une analyse statistique. La corrélation de Pearson entre ces deux colonnes est calculée à l'aide de la méthode .corr() de Pandas. Le résultat, un coefficient compris entre -1 et +1, est affiché dans la console pour indiquer la force et la direction de la relation entre les deux variables (positive, négative ou nulle).
- Ensuite, un graphe de dispersion est créé avec seaborn. Chaque point du graphe représente une observation dans les données, où radiance_16_km^2 est représenté sur l'axe X et radiance sur l'axe Y. Une ligne de régression linéaire rouge est tracée pour illustrer la tendance globale entre les deux variables. La transparence des points (alpha) améliore la lisibilité lorsque de nombreux points se chevauchent. Le graphique est agrémenté d'un titre contenant la valeur de la corrélation et des étiquettes claires pour les axes. Une grille est ajoutée pour faciliter l'interprétation visuelle.
- Enfin, le graphe est affiché à l'aide de plt.show(). Cette visualisation permet de comprendre comment les deux variables sont liées et si une relation linéaire existe entre elles. Ce type d'analyse est particulièrement utile pour explorer des relations potentielles dans des études socio-économiques ou environnementales basées sur des données spatiales et des intensités lumineuses.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le fichier CSV avec les colonnes 'radiance_16_km^2' et 'radiance'
csv_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance.csv"
df = pd.read_csv(csv_file_path)

# Calculer la corrélation entre 'radiance_16_km^2' et 'radiance'
correlation = df[['radiance_16_km^2', 'radiance']].corr().iloc[0, 1]
print(f"Corrélation entre 'radiance_16_km^2' et 'radiance' : {correlation:.2f}")

# Créer un graphe de dispersion
plt.figure(figsize=(10, 6))
sns.regplot(
    x='radiance_16_km^2',
    y='radiance',
    data=df,
    scatter_kws={'alpha': 0.6},
    line_kws={'color': 'red'},
)
plt.title(f"Corrélation entre 'radiance_16_km^2' et 'radiance' (r = {correlation:.2f})", fontsize=14)
plt.xlabel('Radiance 16 km²', fontsize=12)
plt.ylabel('Radiance', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Afficher le graphe
plt.show()


Ce script détecte les doublons dans un fichier CSV en se basant sur les colonnes latitude et longitude. Il charge les données dans un DataFrame, identifie les lignes contenant les mêmes coordonnées GPS en utilisant la méthode duplicated() avec l'option keep=False, et affiche les doublons trouvés ainsi que leur nombre total. Si aucun doublon n'est détecté, il indique qu'il n'y en a pas. Optionnellement, les doublons peuvent être sauvegardés dans un fichier CSV pour une analyse ou un traitement ultérieur. Ce script est utile pour valider et nettoyer des données géospatiales en garantissant l'unicité des points GPS avant une analyse.

In [ ]:
import pandas as pd

# Charger le fichier CSV
csv_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance.csv"
df = pd.read_csv(csv_file_path)

# Vérifier les doublons basés sur 'latitude' et 'longitude'
duplicated_rows = df[df.duplicated(subset=['latitude', 'longitude'], keep=False)]

if not duplicated_rows.empty:
    print("Doublons trouvés dans les coordonnées latitude et longitude :")
    print(duplicated_rows)
    print(f"Nombre total de doublons : {len(duplicated_rows)}")
else:
    print("Aucun doublon trouvé dans les coordonnées latitude et longitude.")

# Sauvegarder les doublons dans un fichier CSV pour analyse
#output_duplicated_file = r"D:\wealth_predict_sentinel\Data\processed_csv\duplicated_lat_lon.csv"
#duplicated_rows.to_csv(output_duplicated_file, index=False)
#print(f"Doublons sauvegardés dans : {output_duplicated_file}")


Ce script réalise une analyse exploratoire de la colonne `radiance_16_km^2` d'un fichier CSV contenant des données géospatiales. Tout d'abord, il charge les données depuis un chemin spécifié et calcule des **statistiques descriptives** de base pour la colonne, incluant des mesures telles que la moyenne, l'écart-type, et les quantiles. Il extrait également un résumé en cinq nombres (minimum, premier quartile, médiane, troisième quartile, maximum) pour résumer la distribution des valeurs.

Ensuite, le script vérifie la présence de **valeurs manquantes** dans la colonne `radiance_16_km^2` en comptant le nombre de cellules vides. Cela permet de s'assurer de la complétude des données ou de signaler des observations manquantes pour un traitement futur.

Il procède ensuite à l'**identification des outliers** en calculant l'écart interquartile (IQR). À l'aide de cette mesure, il définit des bornes inférieure et supérieure pour détecter les valeurs anormalement basses ou élevées dans les données. Les lignes contenant des outliers sont listées avec leurs coordonnées géographiques (latitude et longitude) pour une inspection ou une correction éventuelle.

Enfin, un **histogramme** est généré pour visualiser la distribution des valeurs de `radiance_16_km^2`. Ce graphique aide à comprendre la répartition des données, à identifier des tendances ou des anomalies, et à évaluer si les valeurs suivent une certaine distribution (par exemple, normale ou asymétrique). Le script fournit ainsi un aperçu statistique et visuel complet de la colonne d'intérêt.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier CSV
csv_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance.csv"
df = pd.read_csv(csv_file_path)

# 1. Statistiques descriptives et 5-number summary
descriptive_stats = df['radiance_16_km^2'].describe()
five_number_summary = descriptive_stats[['min', '25%', '50%', '75%', 'max']]
print("Statistiques descriptives :")
print(descriptive_stats)
print("\nRésumé en cinq nombres :")
print(five_number_summary)

# 2. Vérification des valeurs manquantes
missing_values = df['radiance_16_km^2'].isnull().sum()
print(f"\nNombre de valeurs manquantes pour 'radiance_16_km^2' : {missing_values}")

# 3. Détection des outliers
Q1 = df['radiance_16_km^2'].quantile(0.25)
Q3 = df['radiance_16_km^2'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['radiance_16_km^2'] < lower_bound) | (df['radiance_16_km^2'] > upper_bound)]
print("\nListe des outliers (latitude, longitude) :")
print(outliers[['latitude', 'longitude']])
# Calcul de l'IQR
Q1 = df['radiance_16_km^2'].quantile(0.25)
Q3 = df['radiance_16_km^2'].quantile(0.75)
IQR = Q3 - Q1


# Afficher les bornes des outliers
print(f"Les valeurs inférieures à {lower_bound:.2f} ou supérieures à {upper_bound:.2f} sont considérées comme des outliers.")

# 4. Histogramme de 'radiance_16_km^2'
plt.figure(figsize=(10, 6))
plt.hist(df['radiance_16_km^2'].dropna(), bins=30, alpha=0.7, edgecolor='black')
plt.title("Histogramme de 'radiance_16_km^2'", fontsize=14)
plt.xlabel("Radiance 16 km²", fontsize=12)
plt.ylabel("Fréquence", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


#### On supprime toutes les valeurs de df qui sont au-dessus du 99.9ème percentile de la colonne radiance

In [ ]:
# Calcul du 99ème percentile
percentile_99 = df['radiance_16_km^2'].quantile(0.999)
print(f"Le 99ème percentile de la colonne 'radiance_16_km^2' est : {percentile_99}")

# Filtrer le DataFrame pour ne garder que les valeurs de 'radiance_16_km^2' inférieures ou égales au 99ème percentile
initial_count = len(df)
df = df[df['radiance_16_km^2'] <= percentile_99]
filtered_count = len(df)

# Calculer et imprimer le nombre de valeurs supprimées
num_values_removed = initial_count - filtered_count
print(f"Le nombre de valeurs supprimées est : {num_values_removed}")

# Afficher le DataFrame mis à jour
print(df)


Ce script analyse la colonne `radiance_16_km^2` d'un fichier CSV pour fournir un aperçu statistique, vérifier la qualité des données, détecter les valeurs aberrantes et visualiser la distribution des données. Il commence par charger les données du fichier CSV spécifié et calcule des **statistiques descriptives** comme la moyenne, l’écart-type et les quantiles. Un résumé en cinq nombres (minimum, premier quartile, médiane, troisième quartile, maximum) est extrait pour mieux comprendre la distribution des valeurs.

Ensuite, le script vérifie la présence de **valeurs manquantes** dans la colonne `radiance_16_km^2`, en comptant les cellules vides pour s’assurer de la complétude des données. Il détecte également les **outliers** en calculant l’écart interquartile (IQR) et en définissant des bornes basées sur \( Q1 - 1.5 \times IQR \) et \( Q3 + 1.5 \times IQR \). Les valeurs en dehors de ces bornes sont considérées comme des anomalies. Les coordonnées géographiques des outliers (`latitude` et `longitude`) sont affichées si des outliers sont détectés, sinon un message indique qu’aucun outlier n’est présent.

Enfin, un **histogramme** est généré pour visualiser la répartition des valeurs de `radiance_16_km^2`. Ce graphique utilise 30 intervalles pour montrer la fréquence des valeurs et aide à identifier visuellement des tendances, des asymétries ou des pics dans les données. Ce script fournit une analyse exhaustive des données, combinant des résumés statistiques, des vérifications de qualité et des visualisations pour une exploration approfondie.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# 1. Statistiques descriptives et 5-number summary
descriptive_stats = df['radiance_16_km^2'].describe()
five_number_summary = descriptive_stats[['min', '25%', '50%', '75%', 'max']]
print("Statistiques descriptives :")
print(descriptive_stats)
print("\nRésumé en cinq nombres :")
print(five_number_summary)

# 2. Vérification des valeurs manquantes
missing_values = df['radiance_16_km^2'].isnull().sum()
print(f"\nNombre de valeurs manquantes pour 'radiance_16_km^2' : {missing_values}")

# 3. Détection des outliers
Q1 = df['radiance_16_km^2'].quantile(0.25)
Q3 = df['radiance_16_km^2'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['radiance_16_km^2'] < lower_bound) | (df['radiance_16_km^2'] > upper_bound)]

# Affichage des bornes pour les outliers
print(f"\nLes valeurs inférieures à {lower_bound:.2f} ou supérieures à {upper_bound:.2f} sont considérées comme des outliers.")

# Affichage des outliers
print("\nListe des outliers (latitude, longitude) :")
if not outliers.empty:
    print(outliers[['latitude', 'longitude']])
else:
    print("Aucun outlier trouvé.")

# 4. Histogramme de 'radiance_16_km^2'

plt.figure(figsize=(10, 6))
plt.hist(df['radiance_16_km^2'].dropna(), bins=30, alpha=0.7, edgecolor='black')
plt.title("Histogramme de 'radiance_16_km^2'", fontsize=14)
plt.xlabel("Radiance 16 km²", fontsize=12)
plt.ylabel("Fréquence", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

##### sauvegarde

In [ ]:
# Sauvegarder le DataFrame df dans un fichier CSV
output_file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance_filtered.csv"
df.to_csv(output_file_path, index=False)

print(f"Fichier sauvegardé avec succès dans : {output_file_path}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns

# Charger les données
def load_data(file_path):
    """Charge les données à partir d'un fichier CSV."""
    return pd.read_csv(file_path)

# Évaluer les scores BIC
def evaluate_bic(data, column, n_components_range):
    """Évalue les scores BIC pour différents nombres de classes."""
    X = data[column].values.reshape(-1, 1)
    bics = []
    gmms = []

    for n_components in n_components_range:
        gmm = GaussianMixture(n_components=n_components, random_state=42)
        gmm.fit(X)
        gmms.append(gmm)
        bics.append(gmm.bic(X))

    return bics, gmms

# Afficher le graphique des scores BIC
def plot_bic(n_components_range, bics):
    """Affiche le graphique des scores BIC."""
    plt.figure(figsize=(10, 6))
    plt.plot(n_components_range, bics, marker='o', linestyle='-')
    plt.title('Relation entre le nombre de classes et le BIC', fontsize=14)
    plt.xlabel('Nombre de classes', fontsize=12)
    plt.ylabel('Score BIC', fontsize=12)
    plt.grid(True)
    plt.show()

# Appliquer le modèle GMM
def apply_gmm(data, column, gmm):
    """Applique un modèle GMM sur une colonne et ajoute les classes."""
    X = data[column].values.reshape(-1, 1)
    data['class'] = gmm.predict(X)
    return data

# Réordonner les classes
def reorder_classes(data, gmm, column):
    """Réordonne les classes selon les plages de valeurs croissantes."""
    class_ranges = []
    for i in range(gmm.n_components):
        class_values = data[data['class'] == i][column]
        class_ranges.append((i, class_values.min(), class_values.max(), len(class_values)))

    # Trier les classes par leur valeur minimale
    class_ranges_sorted = sorted(class_ranges, key=lambda x: x[1])
    new_class_order = {old: new for new, (old, _, _, _) in enumerate(class_ranges_sorted)}

    # Réattribuer les classes dans le DataFrame
    data['class'] = data['class'].map(new_class_order)

    # Afficher les informations des classes triées
    print("\nPlages de valeurs des classes (triées) :")
    for idx, (_, min_val, max_val, count) in enumerate(class_ranges_sorted, start=1):
        print(f"Classe {idx}: Min = {min_val:.6f}, Max = {max_val:.6f}, Nombre d'observations = {count}")

    return data

# Afficher les histogrammes des classes
def plot_histograms(data, column, cluster_column):
    """Affiche un histogramme montrant la distribution des classes."""
    plt.figure(figsize=(10, 6))
    sns.histplot(data[cluster_column], bins='auto', kde=False)
    plt.title(f"Distribution des classes pour {column}", fontsize=14)
    plt.xlabel("Classe", fontsize=12)
    plt.ylabel("Fréquence", fontsize=12)
    plt.grid(True)
    plt.show()

# Fonction principale
def main(file_path, output_path, n_components_range=range(1, 10)):
    # Charger les données
    data = load_data(file_path)

    # Évaluer les scores BIC
    bics, gmms = evaluate_bic(data, 'radiance_16_km^2', n_components_range)

    # Afficher le graphique des scores BIC
    plot_bic(n_components_range, bics)

    # Sélectionner le modèle GMM avec le BIC minimal
    optimal_index = np.argmin(bics)
    optimal_gmm = gmms[optimal_index]
    optimal_n_components = n_components_range[optimal_index]

    print(f"\nNombre optimal de classes basé sur le BIC : {optimal_n_components}")

    # Appliquer le modèle GMM optimal
    data = apply_gmm(data, 'radiance_16_km^2', optimal_gmm)

    # Réordonner les classes
    data = reorder_classes(data, optimal_gmm, 'radiance_16_km^2')

    # Afficher les histogrammes
    plot_histograms(data, 'radiance_16_km^2', 'class')

    # Sauvegarder le fichier avec les classes
    data.to_csv(output_path, index=False)
    print(f"\nFichier sauvegardé avec succès dans : {output_path}")

    return data

# Chemins des fichiers
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_with_radiance_filtered.csv"
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_classified.csv"

# Lancer l'analyse
df_final = main(file_path, output_path, n_components_range=range(1, 5))

# Afficher les premières lignes du DataFrame final
print(df_final.head())


Ce script commence par charger un fichier CSV contenant des données classifiées, où chaque observation est associée à une classe et une valeur de radiance (radiance_16_km^2). Il regroupe ensuite les données par classe et calcule des statistiques descriptives pour chaque classe, à savoir la valeur minimale, la valeur maximale, et le nombre total d'observations.

Les résultats sont organisés dans un nouveau DataFrame class_stats pour faciliter l'analyse. Pour chaque classe, les statistiques calculées sont affichées de manière lisible : la classe, la valeur minimale et maximale de radiance, ainsi que le nombre d'observations dans la classe.

In [ ]:
import pandas as pd

# Charger le fichier CSV
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_classified.csv"
df = pd.read_csv(file_path)

# Calculer les minimums et maximums pour chaque classe
class_stats = df.groupby('class')['radiance_16_km^2'].agg(['min', 'max', 'count']).reset_index()

# Afficher les résultats
print("Maximum et Minimum pour chaque classe :\n")
for _, row in class_stats.iterrows():
    print(f"Classe {int(row['class'])}: Min = {row['min']:.6f}, Max = {row['max']:.6f}, Nombre d'observations = {int(row['count'])}")


Ce script analyse un fichier CSV contenant des données classifiées par classe et extrait des informations pertinentes. Voici ce qu'il fait :

1. **Chargement des données** : Le script commence par charger un fichier CSV à partir du chemin spécifié, contenant des colonnes telles que `class` et `radiance_16_km^2`.

2. **Calcul des statistiques par classe** : À l’aide de la méthode `groupby`, il regroupe les données par classe et calcule le minimum, le maximum, et le nombre d’observations (`count`) pour chaque classe dans la colonne `radiance_16_km^2`. Ces statistiques sont stockées dans une DataFrame nommée `class_stats`.

3. **Affichage des statistiques** : Les statistiques calculées sont affichées de manière lisible pour chaque classe, indiquant les valeurs minimales et maximales de `radiance_16_km^2`, ainsi que le nombre total d’observations dans chaque classe.

4. **Exemple de lignes par classe** : Pour chaque classe, le script extrait et affiche les deux premières lignes correspondantes. Cela permet de visualiser un échantillon des données disponibles pour chaque classe.

En résumé, ce script fournit une vue d’ensemble des classes en termes de statistiques descriptives et présente un échantillon des données pour mieux comprendre leur distribution.

In [ ]:
import pandas as pd

# Charger le fichier CSV
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_classified.csv"
df = pd.read_csv(file_path)

# Calculer les statistiques des classes
class_stats = df.groupby('class')['radiance_16_km^2'].agg(['min', 'max', 'count']).reset_index()

# Afficher les statistiques des classes
print("Maximum et Minimum pour chaque classe :\n")
for _, row in class_stats.iterrows():
    print(f"Classe {int(row['class'])}: Min = {row['min']:.6f}, Max = {row['max']:.6f}, Nombre d'observations = {int(row['count'])}")

# Imprimer 2 lignes de chaque classe
print("\nExemples de 2 lignes par classe :\n")
for class_label in df['class'].unique():
    sample_rows = df[df['class'] == class_label].head(2)
    print(f"Classe {class_label} :\n", sample_rows, "\n")


Ce script effectue un échantillonnage équilibré des données par classe et concatène les échantillons pour former un nouveau DataFrame. Voici une description compacte des différentes étapes :

1. **Sélection par classe** : Le script commence par sélectionner les données correspondant à chaque classe (`class` égale à 0, 1, 2, et 3) dans le DataFrame initial.

2. **Échantillonnage aléatoire** : Pour chaque classe, un échantillon aléatoire de 100 000 lignes est extrait à l'aide de la méthode `sample`. Le paramètre `random_state` est utilisé pour garantir que les résultats soient reproductibles.

3. **Concaténation des échantillons** : Les échantillons des différentes classes sont fusionnés en un nouveau DataFrame appelé `df_sampled`. La concaténation se fait en ignorant les index initiaux afin de réindexer le nouveau DataFrame.

4. **Vérification du résultat** : Le script affiche les 15 premières lignes du nouveau DataFrame (`df_sampled`) pour examiner un aperçu des données. Il imprime également le nombre d’observations pour chaque classe afin de vérifier que l’échantillonnage a été équilibré.

En résumé, ce script extrait des échantillons équilibrés de données pour chaque classe, les fusionne dans un nouveau DataFrame, et vérifie visuellement et quantitativement le résultat.

In [ ]:
import pandas as pd
import random

# Définir une graine aléatoire
seed = 42
random.seed(seed)

# Charger le fichier CSV
df = pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_classified.csv")

# Création d'un échantillon de 95 000 lignes pour chaque valeur de 'class'
df_class_0 = df[df['class'] == 0].sample(n=95000, random_state=seed)
df_class_1 = df[df['class'] == 1].sample(n=95000, random_state=seed)
df_class_2 = df[df['class'] == 2].sample(n=95000, random_state=seed)
df_class_3 = df[df['class'] == 3].sample(n=95000, random_state=seed)

# Concatenation des échantillons pour former le nouveau DataFrame
df_sampled = pd.concat([df_class_0, df_class_1, df_class_2, df_class_3], ignore_index=True)

# Affichage des premières lignes pour vérifier le résultat
print(df_sampled.head(15))
print(df_sampled['class'].value_counts())

# Sauvegarde de df_sampled au format CSV
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_380k_obs.csv"
df_sampled.to_csv(output_path, index=False)

print(f"Le fichier a été sauvegardé avec succès à l'emplacement : {output_path}")


In [ ]:
import pandas as pd
import random

# Définir une graine aléatoire
seed = 42
random.seed(seed)

# Charger le fichier CSV
df = pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\df_afrique_pharmacies_schools_combined_sampled_classified.csv")

# Création d'un échantillon de 95 000 lignes pour chaque valeur de 'class'
df_class_0 = df[df['class'] == 0].sample(n=2000, random_state=seed)
df_class_1 = df[df['class'] == 1].sample(n=2000, random_state=seed)
df_class_2 = df[df['class'] == 2].sample(n=2000, random_state=seed)
df_class_3 = df[df['class'] == 3].sample(n=2000, random_state=seed)

# Concatenation des échantillons pour former le nouveau DataFrame
df_sampled_2 = pd.concat([df_class_0, df_class_1, df_class_2, df_class_3], ignore_index=True)

# Affichage des premières lignes pour vérifier le résultat
print(df_sampled_2.head(15))
print(df_sampled_2['class'].value_counts())

# Sauvegarde de df_sampled au format CSV
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_8k_obs.csv"
df_sampled_2.to_csv(output_path, index=False)

print(f"Le fichier a été sauvegardé avec succès à l'emplacement : {output_path}")


In [ ]:
import pandas as pd

# Charger le fichier CSV
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_8k_obs.csv"
df1 = pd.read_csv(file_path)

# Calculer les statistiques des classes
class_stats = df1.groupby('class')['radiance_16_km^2'].agg(['min', 'max', 'count']).reset_index()

# Afficher les statistiques des classes
print("Maximum et Minimum pour chaque classe :\n")
for _, row in class_stats.iterrows():
    print(f"Classe {int(row['class'])}: Min = {row['min']:.6f}, Max = {row['max']:.6f}, Nombre d'observations = {int(row['count'])}")

# Imprimer 2 lignes de chaque classe
print("\nExemples de 2 lignes par classe :\n")
for class_label in df1['class'].unique():
    sample_rows = df1[df1['class'] == class_label].head(2)  # Correction ici
    print(f"Classe {class_label} :\n", sample_rows, "\n")


In [ ]:
!pip install folium

Ce script vise à créer deux types de cartes de chaleur représentant les intensités lumineuses nocturnes, à partir d’un fichier CSV contenant des données géospatiales. Ces cartes sont ensuite sauvegardées dans des répertoires spécifiques.
### Chargement des données
Le script commence par charger un fichier CSV situé à une adresse spécifiée, contenant des colonnes telles que `latitude`, `longitude` et `radiance`. Ces données servent à générer des visualisations basées sur la répartition géographique des intensités lumineuses.
### Carte de chaleur interactive
La première fonction, **`create_interactive_heatmap`**, utilise la bibliothèque `folium` pour générer une carte de chaleur interactive. Cette carte est centrée sur les coordonnées moyennes des points géographiques (latitude et longitude). Chaque point est pondéré par sa valeur de radiance, permettant d’identifier visuellement les zones à forte intensité lumineuse. La carte est sauvegardée au format HTML dans le répertoire spécifié (`html_output_dir`), offrant une interface interactive pour explorer les données.
### Carte de chaleur statique
La seconde fonction, **`create_static_heatmap`**, génère une carte de chaleur statique à l’aide de `matplotlib`. Cette carte utilise un nuage de points où la couleur reflète l’intensité lumineuse selon un dégradé de la palette `inferno`. Une barre de couleur associée permet d’interpréter les valeurs de radiance. Cette visualisation est sauvegardée au format PNG dans le répertoire spécifié (`figures_output_dir`), fournissant une image de haute qualité pour une utilisation hors ligne.
### Résultat
Le script produit une carte interactive au format HTML et une carte statique au format PNG, chacune illustrant la répartition géographique des intensités lumineuses nocturnes. Ces fichiers sont sauvegardés dans des répertoires définis, facilitant leur partage et leur utilisation dans d’autres analyses.

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt

# Charger les données
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_8k_obs.csv"
df = pd.read_csv(file_path)

# Emplacements de sauvegarde
html_output_dir = r"D:\wealth_predict_sentinel\Data\html_output"
figures_output_dir = r"D:\wealth_predict_sentinel\figures_graphs"

# Carte 1 : Carte de chaleur interactive (HTML)
def create_interactive_heatmap(df, output_path):
    """Créer une carte interactive de chaleur et la sauvegarder au format HTML."""
    m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=6)
    heat_data = [[row['latitude'], row['longitude'], row['radiance_16_km^2']] for index, row in df.iterrows()]
    HeatMap(heat_data).add_to(m)
    m.save(output_path)
    # Afficher la carte
    m
    print(f"Carte de chaleur interactive sauvegardée dans : {output_path}")
    
# Générer et sauvegarder la carte interactive
interactive_map_path = f"{html_output_dir}\heatmap_radiance.html"
create_interactive_heatmap(df, interactive_map_path)

# Carte 2 : Carte de chaleur statique (Image)
def create_static_heatmap(df, output_path):
    """Créer une carte de chaleur statique et la sauvegarder en tant qu'image."""
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(df['longitude'], df['latitude'], c=df['radiance_16_km^2'], cmap='inferno', alpha=0.7)
    plt.colorbar(scatter, label='Radiance')
    plt.title('Carte de Chaleur des Intensités Lumineuses Nocturnes')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.savefig(output_path, format='png', dpi=300)
    plt.show()
    print(f"Carte de chaleur statique sauvegardée dans : {output_path}")

# Générer et sauvegarder la carte statique
static_map_path = f"{figures_output_dir}\heatmap_radiance_16_km^2.png"
create_static_heatmap(df, static_map_path)


In [ ]:
!pip install geopandas

Ce script produit une **carte de chaleur statique** des intensités lumineuses nocturnes sur le continent africain en utilisant des données GPS et un fichier GeoJSON des frontières de l'Afrique. Voici ce qu'il fait, résumé en étapes clés :
#### 1. **Chargement des données** :
   - Le script charge un fichier CSV contenant les données GPS (`latitude`, `longitude`) et les valeurs de radiance (`radiance_16_km^2`) dans un DataFrame `pandas`.
   - Il charge également un fichier GeoJSON contenant les frontières géographiques du continent africain en utilisant `geopandas`.
#### 2. **Création de la carte** :
   - Une carte de base est générée en affichant les frontières de l'Afrique avec des zones grises et des contours noirs pour délimiter le continent.
   - Les points GPS issus des données CSV sont superposés à cette carte sous forme de nuage de points, où chaque point est coloré selon la valeur de radiance. Une palette de couleurs (`inferno`) est utilisée pour refléter l'intensité lumineuse (valeurs élevées représentées par des couleurs chaudes, comme le jaune, et des valeurs faibles par des couleurs froides, comme le violet).
#### 3. **Ajout de la légende** :
   - Une barre de couleur est ajoutée à la carte pour permettre l'interprétation des valeurs de radiance. Elle indique comment les couleurs des points se traduisent en intensités lumineuses.
#### 4. **Personnalisation de la carte** :
   - La carte est enrichie d'un titre décrivant son contenu ("Carte de Chaleur des Intensités Lumineuses Nocturnes (Afrique)").
   - Les axes de la carte sont étiquetés pour indiquer les coordonnées géographiques (longitude et latitude).
#### 5. **Sauvegarde et affichage** :
   - La carte générée est sauvegardée au format PNG avec une haute résolution (300 DPI) dans un répertoire spécifié (`D:\wealth_predict_sentinel\figures_graphs`).
   - Enfin, la carte est affichée pour une visualisation immédiate.
Ce script est utile pour analyser la répartition spatiale des intensités lumineuses sur le continent africain, mettant en évidence les zones urbaines ou densément peuplées grâce aux données de radiance.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Chemins des fichiers
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_8k_obs.csv"
geojson_path = r"D:\wealth_predict_sentinel\Data\downloaded\africa.json"
figures_output_dir = r"D:\wealth_predict_sentinel\figures_graphs"

# Charger les données GPS
print("Chargement des données GPS...")
df = pd.read_csv(file_path)

# Charger les frontières de l'Afrique à partir du GeoJSON
print("Chargement des frontières de l'Afrique depuis le fichier GeoJSON...")
africa = gpd.read_file(geojson_path)

# Créer la figure et les axes
print("Création de la carte...")
fig, ax = plt.subplots(figsize=(12, 12))

# Dessiner les frontières de l'Afrique
africa.plot(ax=ax, color='lightgrey', edgecolor='black')

# Ajouter les points GPS avec une carte de couleur basée sur la radiance
scatter = ax.scatter(
    df['longitude'], 
    df['latitude'], 
    c=df['radiance_16_km^2'], 
    cmap='inferno', 
    alpha=0.7, 
    s=10
)

# Ajouter une barre de couleur
cbar = plt.colorbar(scatter, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('radiance', fontsize=12)

# Ajouter les titres et les étiquettes
ax.set_title('Carte de Chaleur des Intensités Lumineuses Nocturnes (Afrique) 8000 emplacements', fontsize=16)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

# Sauvegarder la carte
output_path = f"{figures_output_dir}\heatmap_radiance_africa.png"
plt.savefig(output_path, format='png', dpi=300)
print(f"Carte sauvegardée avec succès dans : {output_path}")

# Afficher la carte
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Chemins des fichiers
file_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_380k_obs.csv"
geojson_path = r"D:\wealth_predict_sentinel\Data\downloaded\africa.json"
figures_output_dir = r"D:\wealth_predict_sentinel\figures_graphs"

# Charger les données GPS
print("Chargement des données GPS...")
df = pd.read_csv(file_path)

# Charger les frontières de l'Afrique à partir du GeoJSON
print("Chargement des frontières de l'Afrique depuis le fichier GeoJSON...")
africa = gpd.read_file(geojson_path)

# Créer la figure et les axes
print("Création de la carte...")
fig, ax = plt.subplots(figsize=(12, 12))

# Dessiner les frontières de l'Afrique
africa.plot(ax=ax, color='lightgrey', edgecolor='black')

# Ajouter les points GPS avec une carte de couleur basée sur la radiance
scatter = ax.scatter(
    df['longitude'], 
    df['latitude'], 
    c=df['radiance_16_km^2'], 
    cmap='inferno', 
    alpha=0.7, 
    s=10
)

# Ajouter une barre de couleur
cbar = plt.colorbar(scatter, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Radiance', fontsize=12)

# Ajouter les titres et les étiquettes
ax.set_title('Carte de Chaleur des Intensités Lumineuses Nocturnes (Afrique) - 380,000 Emplacements', fontsize=16)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

# Sauvegarder la carte
output_path = f"{figures_output_dir}\heatmap_radiance_africa_380k.png"
plt.savefig(output_path, format='png', dpi=300)
print(f"Carte sauvegardée avec succès dans : {output_path}")

# Afficher la carte
plt.show()